# 설비마스터에서 소둔로 한 행을 찾아 온도.csv에 붙이기

**담당: 박민호**  ·  서브 이슈: #45  ·  상위 이슈: #43

| 항목 | 값 |
| --- | --- |
| 브랜치 | `feat/45-equipment-master-namelesspark` |
| 노트북 | `notebooks/04_equipment_master_timeseries_join_namelesspark.ipynb` |
| 대상 데이터 | `data/G-01_설비마스터.csv`, `data/T-CR1-CAL01_온도.csv` |

## 결론 (실행 결과로 확인)

- 설비마스터 104행 중 우리 설비는 **`P1-CR1-CAL01`(연속 소둔로) 한 행**뿐임 (CSV 93번째 줄)
- 두 파일을 잇는 기준 컬럼은 `EQP_TAG`. 온도.csv에는 설비 컬럼이 없으므로 **파일 이름의 `CAL01`로 그 한 행을 지정**함
- 그 한 행을 온도 전체에 붙이는 것이므로 시계열 조인이 아니라 **상수 붙이기**. 행 수는 변하지 않음
- 이 노트북의 실질적 목적: **`G-02_정비이력.csv`를 붙일 때 쓸 `EQP_TAG` 값을 확정하는 것**

## 확인할 것 세 가지

1. 소둔로 한 행을 제대로 집었는지, 그 행에 빈 값이 있는지
2. 온도.csv에 붙였을 때 행 수가 그대로인지 (안 불어나는지)
3. `INSTALL_DT <= MEAS_DT` 조건이 실제로 쓸모가 있는지

> 공개 저장소이므로 원본 데이터 행은 출력하지 않음. 설비마스터의 대상 1행은 분석 근거라 필드만 출력함

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
FIG_DIR = os.path.join("..", "figures")        # 그래프 저장 위치
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")

ENC = "utf-8-sig"                              # 마스터 CSV는 BOM 있음

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# 그래프 한글 깨짐 방지 - 설치된 폰트 중 있는 것을 골라 rcParams에 넣습니다
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False         # 음수 기호가 □ 로 나오는 것 방지
rcParams["figure.dpi"] = 110

print("경로 확인:", os.path.exists(TEMP_CSV), os.path.exists(EQP_CSV))


def show_kv(d):
    """숫자와 참/거짓 같은 값만 key value 형태로 출력합니다."""
    for k, v in d.items():
        print(f"{k:<30} {v}")

## 설비마스터.csv 컬럼 뜻

컬럼 12개. 뜻은 약어와 실제 값의 형태로 읽은 것이고, 단위 컬럼은 파일에 없음

| 컬럼 | 뜻 | 값의 형태 | 대상 설비(소둔로) 값 |
| --- | --- | --- | --- |
| `EQP_TAG` | 설비 식별자. 이 파일의 기준 컬럼 | `P1-공장-설비코드` | `P1-CR1-CAL01` |
| `EQP_NM` | 설비 이름 | 한글 | 연속 소둔로 |
| `PARENT_TAG` | 상위 설비의 `EQP_TAG` | `EQP_TAG` 값 (루트는 빈칸) | `P1-CR1` (냉연공장) |
| `PLANT_CD` | 공장·라인 코드 | BF1, BF2, SM1, CC1, CC2, CR1, SP1, SP2 (8종) | `CR1` |
| `EQP_TYPE` | 설비 종류 | 라인, 연속로, 압연기, 펌프 등 33종 | 연속로 |
| `MAKER` | 제작사 | 한글 (32행 빈칸) | 동성기계공업 |
| `INSTALL_DT` | 설치일. 이 파일의 유일한 시간 컬럼 | 날짜 (1985 ~ 2021) | 2014-05-31 |
| `RATED_KW` | 정격 출력 | 5 ~ 22000 (32행 빈칸) | 1200 |
| `RATED_RPM` | 정격 회전수 | 12 ~ 1780 (39행 빈칸) | **빈칸** |
| `CRIT_LVL` | 설비 중요도 등급 | A, B, C | `A` (최상) |
| `CBM_TGT_YN` | 상태기반정비(CBM) 대상 여부 | Y, N | `Y` |
| `DATA_AVL_YN` | 계측 데이터 보유 여부 | Y, N | `Y` |

이 노트북에서 실제로 쓰는 컬럼

| 쓰는 컬럼 | 용도 |
| --- | --- |
| `EQP_TAG`, `PLANT_CD` | 소둔로 한 행을 집고, 온도.csv에 붙일 때 기준으로 씀 |
| `INSTALL_DT` | `INSTALL_DT <= MEAS_DT` 조건 검증 |
| `EQP_NM`, `EQP_TYPE`, `CRIT_LVL`, `CBM_TGT_YN`, `DATA_AVL_YN` | 대상 설비가 맞는지 확인하는 근거 |
| `RATED_KW`, `RATED_RPM`, `MAKER`, `PARENT_TAG` | 안 씀. 참고용 |

## 1단계 - 소둔로 한 행 집기

- 온도 파일 이름이 `T-CR1-CAL01_온도.csv` → 라인 `CR1`, 설비 코드 `CAL01`
- 이 두 조건으로 설비마스터를 걸러서 **1건으로 떨어지는지** 확인
    - 2건 이상이면 설비를 잘못 집은 것이므로 `assert`로 멈춤
- 집은 행에 빈 값이 있으면 그 컬럼은 조인 후에도 못 씀

In [ ]:
eqp = pd.read_csv(EQP_CSV, encoding=ENC)
eqp["INSTALL_DT"] = pd.to_datetime(eqp["INSTALL_DT"], errors="coerce")

fname = os.path.basename(TEMP_CSV)                       # T-CR1-CAL01_온도.csv
plant, eqp_code = fname.split("_")[0].split("-")[1:3]    # ['CR1', 'CAL01']

cand = eqp[(eqp["PLANT_CD"] == plant) & (eqp["EQP_TAG"].str.endswith(eqp_code))]
assert len(cand) == 1, "설비가 1건으로 확정되지 않았습니다. 규칙을 다시 보세요."

row = cand.iloc[0]
EQP_TAG_TARGET = row["EQP_TAG"]

show_kv(
    {
        "설비마스터 전체": f"{len(eqp)} 행 x {eqp.shape[1]} 열",
        "EQP_TAG 유일성": eqp["EQP_TAG"].is_unique,      # 유일해야 조인에서 행이 안 불어남
        "파일명에서 뽑은 조건": f"{plant} + {eqp_code}",
        "찾은 설비 수": len(cand),
        "CSV 줄번호(헤더 포함)": int(cand.index[0]) + 2,
    }
)

print("\n[대상 설비]")
show_kv({k: row[k] for k in ["EQP_TAG", "EQP_NM", "PARENT_TAG", "EQP_TYPE",
                             "MAKER", "INSTALL_DT", "CRIT_LVL", "CBM_TGT_YN", "DATA_AVL_YN"]})

print("\n이 행의 빈 컬럼:", [k for k in eqp.columns if pd.isna(row[k])])

## 2단계 - 온도.csv에 붙여도 행이 안 불어나는지

- 조인 판정에는 두 컬럼만 필요하므로 그것만 읽음 (34만 행 전체를 메모리에 안 올림)

| 온도.csv 컬럼 | 뜻 |
| --- | --- |
| `MEAS_DT` | 계측 시각. 1초 주기 |
| `LOT_NO` | 로트 번호. 강종변경으로 시작되는 연속 생산 구간 (14개) |

- 온도 행 전체에 `EQP_TAG`를 같은 값으로 채우고 설비마스터를 left join

| 확인 항목 | 정상이면 | 어긋나면 |
| --- | --- | --- |
| 행 증폭 배수 | 1.0 (= 조인 후 행 수 / 조인 전 행 수) | 설비마스터에 같은 `EQP_TAG`가 2행 이상 있어 온도 행이 복제됨 |
| 설비 정보가 붙은 행 | 100% | `EQP_TAG` 값을 잘못 만들었음 |

- `validate="m:1"` 은 설비마스터에 같은 `EQP_TAG`가 2개 이상이면 pandas가 에러를 내게 하는 안전장치
- 비교용으로 `PLANT_CD`(라인)로 붙였다면 몇 배가 되는지도 계산해 둠 (실행은 안 함)

In [ ]:
temp = pd.read_csv(TEMP_CSV, usecols=["MEAS_DT", "LOT_NO"], encoding=ENC)
temp["MEAS_DT"] = pd.to_datetime(temp["MEAS_DT"])

n_before = len(temp)
temp["EQP_TAG"] = EQP_TAG_TARGET                                    # 온도 행 전체에 같은 설비 태그
joined = temp.merge(eqp, on="EQP_TAG", how="left", validate="m:1")   # 설비마스터 1행이 붙음

fanout = int((eqp["PLANT_CD"] == plant).sum())                       # 라인으로 붙였을 때의 배수

show_kv(
    {
        "온도 측정 구간": f"{temp['MEAS_DT'].min()} ~ {temp['MEAS_DT'].max()}",
        "LOT_NO 개수": temp["LOT_NO"].nunique(),
        "조인 전 행 수": f"{n_before:,}",
        "조인 후 행 수": f"{len(joined):,}",
        "행 증폭 배수": round(len(joined) / n_before, 4),
        "설비 정보가 붙은 행(%)": round(joined["EQP_NM"].notna().mean() * 100, 2),
        "설비 정보가 못 붙은 행": int(joined["EQP_NM"].isna().sum()),
        "PLANT_CD로 붙였다면": f"{fanout}배 = {n_before * fanout:,} 행",
    }
)

## 3단계 - `INSTALL_DT` 조건의 범위와 한계

- `INSTALL_DT`는 설치된 **한 번의 과거 시점** (변경 이력이 아님)
- 설치가 측정보다 한참 앞이면 `INSTALL_DT <= MEAS_DT`는 항상 참 → 걸러지는 행이 0
- 종료일·변경일 컬럼이 없고 설비당 행이 1개면, 이 파일은 **지금 상태만 담은 스냅샷**
    - 2024년 당시 속성인지 확인할 방법이 없음 → 한계로 기록

In [ ]:
valid = joined["INSTALL_DT"] <= joined["MEAS_DT"]

END_HINTS = ["END", "EXP", "CLOSE", "CHG", "UPD", "VALID", "EFF", "SCRAP", "REMOV"]
end_like = [c for c in eqp.columns if any(h in c.upper() for h in END_HINTS)]

show_kv(
    {
        "대상 설비 설치일": row["INSTALL_DT"].date(),
        "설치 후 경과(측정 시작 기준)": f"{(temp['MEAS_DT'].min() - row['INSTALL_DT']).days / 365.25:.1f} 년",
        "INSTALL_DT <= MEAS_DT 만족(%)": round(valid.mean() * 100, 2),
        "조건에서 탈락한 행": int((~valid).sum()),
        "설치일 결측 설비 수": int(eqp["INSTALL_DT"].isna().sum()),
        "종료·변경일 컬럼": end_like if end_like else "없음",
        "설비당 행 수(최대)": int(eqp.groupby("EQP_TAG").size().max()),
    }
)

### 그래프 - 설치일과 측정 구간의 거리

- 주황 점(소둔로 설치일)이 파란 띠(측정 구간)보다 한참 왼쪽이면, 위 만족 비율 100%가 당연하다는 뜻

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.6))

ax.scatter(eqp["INSTALL_DT"], np.zeros(len(eqp)), s=16, alpha=0.3, color="#9ECAE1",
           label=f"전체 설비 설치일 ({len(eqp)}개)")
ax.scatter([row["INSTALL_DT"]], [0], s=110, color="#F58518", zorder=5,
           label=f"{EQP_TAG_TARGET} 설치일")
ax.axvspan(temp["MEAS_DT"].min(), temp["MEAS_DT"].max(), color="#4C78A8", alpha=0.25,
           label="온도 측정 구간")

ax.set_yticks([])
ax.set_xlabel("연도")
ax.set_title("설비 설치 시점 vs 온도 측정 구간")
ax.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "04_install_vs_measure_window.png"), bbox_inches="tight")
plt.show()

## 결과 정리 - 실행 결과 숫자로 채우기

| 항목 | 확인한 값 |
| --- | --- |
| 설비마스터 크기 | 104행 12열 |
| `EQP_TAG` 유일성 | 유일함 |
| 대상 설비 | P1-CR1-CAL01 |
| 대상 설비 CSV 줄번호 | 93행 |
| 대상 행의 빈 컬럼 | RATED_RPM |
| 조인 전 → 후 행 수 | 345,390 -> 345,390 |
| 행 증폭 배수 | 1배수 |
| 설비 정보가 못 붙은 행 | 다 붙어 있음 |
| `INSTALL_DT <= MEAS_DT` 만족 비율 | 100% |

## 결론

**1. 이어 붙일 기준 컬럼은 `EQP_TAG` 하나**

- 온도.csv에는 설비를 가리키는 컬럼이 없음. 파일 이름의 `CAL01`이 유일한 단서
- 그 값으로 설비마스터에서 소둔로 1행을 지정하고, 온도 전체에 같은 태그를 넣어 `m:1` 조인

**2. 시계열 조인이 아니라 상수 붙이기**

- 온도 34만 행에 설비 1행이 똑같이 붙음. 행 증폭 없고, 붙는 값도 시간에 따라 안 변함
- 그래서 모델 입력 파생변수로는 쓸모 없음
- 쓸모는 두 가지
    - 설비 메타데이터 (설비명, 중요도 A, CBM 대상, 설치일)
    - **`G-02_정비이력.csv`를 붙일 기준 컬럼(`EQP_TAG`) 확보** ← 이게 실질적 목적

**3. `INSTALL_DT` 조건은 정합성 가드까지만**

- 설치일이 측정 구간보다 한참 앞이라 조건이 항상 참 → 분석 구간을 나누는 기준으로는 못 씀
- 쓸 수 있는 용도는 하나: 설치 전 측정값이 있으면 데이터 오류로 잡는 검사
- `설비 사용연수 = MEAS_DT - INSTALL_DT` 도 측정 구간 안에서는 거의 상수라 파생변수로 부적합

**4. 이력 조인은 불가 (스냅샷)**

- 설비당 1행 + 종료일 없음 → 시점별 속성을 붙일 수 없음
- 필요한 것: `EFF_STRT_DT`, `EFF_END_DT`, 설비당 여러 행(버전), 센서 단위 컬럼